# Analitik Hasil Audit Teknologi Industri — Empat Jenis Analitik
Mengikuti alur Praktikum AVD Minggu 1 (deskriptif, diagnostik, prediktif, preskriptif), tetapi disesuaikan untuk **data teks/kualitatif**.
Unggah `fixed-1_-hasil-audit-teknologi-industri.xlsx` ke panel Files Colab sebelum menjalankan sel.

**Catatan penting:** dataset hanya berisi 3 audit dan sebagian besar kolomnya teks bebas, sehingga regresi atau tren angka seperti di notebook penjualan tidak bermakna. Sebagai gantinya dipakai pengkodean kata kunci (keyword coding) dan kemiripan teks.

## Langkah 1 — Impor pustaka dan memuat data

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_excel("fixed-1_-hasil-audit-teknologi-industri.xlsx", sheet_name=0)

# Nama kolom asli sangat panjang, ganti dengan nama pendek
df.columns = ["perusahaan", "kegiatan", "tujuan", "sektor",
              "tahun", "lingkup", "teknologi", "rekomendasi"]
df["id"] = ["A" + str(i + 1) for i in range(len(df))]   # kode singkat tiap audit
df = df.set_index("id")
print("Ukuran data:", df.shape)
df[["perusahaan", "sektor", "tahun"]]

## Langkah 2 — Memahami struktur data (Data Understanding)
Kenali tipe data, nilai kosong, dan panjang teks tiap kolom.

In [ ]:
df.info()
print()
print("Nilai kosong per kolom:")
print(df.isna().sum())
print()
print("Jumlah karakter per kolom teks:")
print(df.drop(columns="tahun").apply(lambda s: s.astype(str).str.len()))

## Langkah 3 — Analitik deskriptif: apa yang terjadi?
Ringkas jumlah audit per sektor dan tahun, jumlah perusahaan terlibat, serta kata yang paling sering muncul.

In [ ]:
# Jumlah perusahaan per audit (dipisah koma)
df["jml_perusahaan"] = df["perusahaan"].str.split(",").str.len()

# Jumlah butir (bullet "- ") pada teknologi eksisting dan rekomendasi
for k in ["lingkup", "teknologi", "rekomendasi"]:
    df["butir_" + k] = df[k].astype(str).str.count(r"(?m)^\s*-\s")

print(df[["sektor", "tahun", "jml_perusahaan",
          "butir_lingkup", "butir_teknologi", "butir_rekomendasi"]])
print()
print("Audit per tahun:\n", df["tahun"].value_counts().sort_index())
print("\nAudit per sektor:\n", df["sektor"].value_counts())

In [ ]:
# Frekuensi kata pada seluruh kolom teks (stopword sederhana)
stop = set("""dan yang di ke dari untuk pada dengan atau dalam ini itu adalah dapat
sudah masih akan oleh serta juga sebagai terkait bagi antara secara hasil""".split())

def kata(teks):
    return [w for w in re.findall(r"[a-zA-Z]{4,}", str(teks).lower()) if w not in stop]

semua = Counter(w for k in ["tujuan", "lingkup", "teknologi", "rekomendasi"]
                for t in df[k] for w in kata(t))
pd.DataFrame(semua.most_common(15), columns=["kata", "frekuensi"])

## Langkah 4 — Analitik diagnostik: mengapa terjadi?
Kita lihat **dimensi teknologi apa saja yang tercakup** pada tiap audit. Kolom lingkup, teknologi, dan rekomendasi digabung lalu dikodekan dengan kamus kata kunci. Ubah kamus ini bila perlu; hasilnya sangat bergantung pada kata kunci yang dipilih.

In [ ]:
kamus = {
    "SDM":              ["sdm", "sumber daya manusia", "tenaga"],
    "Mesin/Peralatan":  ["mesin", "peralatan", "alat", "armada", "reaktor", "fasilitas"],
    "Proses/Produksi":  ["proses", "produksi", "metode", "manufaktur", "operasi"],
    "Manajemen":        ["manajemen", "strategi", "bisnis"],
    "Sistem Informasi": ["sistem informasi", "sisfo", "informasi"],
    "Pemeliharaan":     ["pemeliharaan", "dirawat", "perawatan", "kerusakan"],
    "Bahan Baku":       ["bahan baku"],
    "R&D/Riset":        ["r&d", "riset", "invensi", "inovasi"],
    "Investasi":        ["investasi"],
    "Lingkungan":       ["limbah", "lingkungan"],
}

gabung = (df["lingkup"].astype(str) + " " + df["teknologi"].astype(str)
          + " " + df["rekomendasi"].astype(str)).str.lower()

cakupan = pd.DataFrame(
    {dim: gabung.apply(lambda t: int(any(k in t for k in kws)))
     for dim, kws in kamus.items()})
cakupan["total_dimensi"] = cakupan.sum(axis=1)
print(cakupan)

print("\nDimensi yang tercakup di semua audit:",
      [d for d in kamus if cakupan[d].all()])
print("Dimensi yang hanya muncul di 1 audit:",
      [d for d in kamus if cakupan[d].sum() == 1])

In [ ]:
# Kategori tujuan audit (juga berbasis kata kunci)
tujuan_kat = {
    "Peningkatan kinerja":    ["performance", "peningkatan", "improvement"],
    "Evaluasi aset teknologi":["evaluasi", "mengevaluasi", "asset"],
    "Perencanaan teknologi":  ["perencanaan"],
    "Kerja sama/alih tek.":   ["kerjasama", "kerja sama", "alih teknologi"],
}
t = df["tujuan"].astype(str).str.lower()
tujuan_df = pd.DataFrame({k: t.apply(lambda s: int(any(w in s for w in v)))
                          for k, v in tujuan_kat.items()})
print(tujuan_df)

## Langkah 5 — Visualisasi cakupan dimensi audit
Heatmap sederhana: sel gelap berarti dimensi tersebut tercakup pada audit.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.2))
mat = cakupan.drop(columns="total_dimensi")
ax.imshow(mat.values, cmap="Blues", aspect="auto", vmin=0, vmax=1.3)
ax.set_xticks(range(mat.shape[1]))
ax.set_xticklabels(mat.columns, rotation=40, ha="right", fontsize=8)
ax.set_yticks(range(len(mat)))
ax.set_yticklabels([f"{i} ({df.loc[i, 'sektor']}, {df.loc[i, 'tahun']})" for i in mat.index],
                   fontsize=8)
ax.set_title("Cakupan Dimensi pada Tiap Audit Teknologi")
plt.tight_layout()
plt.show()

# Grafik batang: jumlah dimensi per audit
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(cakupan.index, cakupan["total_dimensi"], color="#3b6ea5")
ax.set_ylabel("Jumlah dimensi tercakup")
ax.set_title("Keluasan Cakupan Audit")
ax.grid(axis="y", alpha=0.3)
plt.show()

## Langkah 6 — Analitik prediktif: apa yang akan terjadi?
Dengan hanya 3 audit tanpa variabel target numerik, **model prediksi tidak dapat dilatih secara valid**. Pendekatan yang layak adalah *kemiripan antar audit* (TF-IDF + cosine similarity), misalnya untuk menebak audit lama mana yang paling relevan sebagai rujukan bila ada proyek audit baru. Ini ilustrasi saja; teknik prediktif sesungguhnya dibahas pada Minggu 13.

In [ ]:
teks = (df["tujuan"].astype(str) + " " + df["lingkup"].astype(str)
        + " " + df["teknologi"].astype(str)).str.lower()

tfidf = TfidfVectorizer(token_pattern=r"[a-zA-Z]{4,}")
X = tfidf.fit_transform(teks)
sim = pd.DataFrame(cosine_similarity(X), index=df.index, columns=df.index).round(2)
print(sim)

# Contoh: rujukan paling mirip untuk proyek audit baru
baru = ["audit teknologi sistem pemeliharaan dan operasi armada transportasi"]
skor = cosine_similarity(tfidf.transform(baru), X)[0]
print("\nAudit paling relevan untuk proyek baru:",
      df.index[skor.argmax()], "-", df.loc[df.index[skor.argmax()], "kegiatan"].strip())

## Langkah 7 — Analitik preskriptif: apa yang sebaiknya dilakukan?
Aturan sederhana: dimensi yang **tidak tercakup** pada suatu audit direkomendasikan untuk ditinjau pada audit lanjutan, dan audit yang masih berstatus "dalam proses" diberi tindak lanjut khusus.

In [ ]:
prioritas = ["SDM", "Mesin/Peralatan", "Proses/Produksi", "Manajemen",
             "Sistem Informasi", "Investasi"]   # dimensi inti, bisa diubah

def rekomendasi(id_):
    celah = [d for d in prioritas if cakupan.loc[id_, d] == 0]
    status = "dalam proses" in str(df.loc[id_, "rekomendasi"]).lower()
    if status:
        aksi = "Selesaikan audit & finalisasi rekomendasi"
    elif celah:
        aksi = "Tinjau dimensi: " + ", ".join(celah)
    else:
        aksi = "Cakupan lengkap, lanjut ke implementasi rekomendasi"
    return aksi

hasil = pd.DataFrame({
    "kegiatan": df["kegiatan"].str.replace("\n", " ").str.strip(),
    "tahun": df["tahun"],
    "tindak_lanjut": [rekomendasi(i) for i in df.index]})
pd.set_option("display.max_colwidth", 80)
print(hasil.to_string())